In [1]:
import os
from process_data import process_data
from prepare_data_for_training import prepare_data
from create_model import create_model
from optimize_model import optimize_and_convert_model,get_gzipped_model_size
import matplotlib.pyplot as plt
import math


In [ ]:
data_directory = 'RowData'
wave_and_idle_data = process_data(row_data_dir=data_directory, mode='csv')


In [ ]:
# plot T second of data for eac label, there is a time column so X axis is time, Y axis is the value of the sensors
T = 5  # seconds
for label in wave_and_idle_data['label'].unique():
    subset = wave_and_idle_data[wave_and_idle_data['label'] == label]
    time = subset['time'][:T*100]  # Assuming data is sampled at 100 Hz
    plt.figure(figsize=(5, 3))
    plt.plot(time, subset['x'][:T*100], label='X-axis')
    plt.plot(time, subset['y'][:T*100], label='Y-axis')
    plt.plot(time, subset['z'][:T*100], label='Z-axis')
    plt.title(f'Sensor Data for Label: {label}')
    plt.xlabel('Time (s)')
    plt.ylabel('Sensor Values')
    plt.legend()
    plt.show()


In [ ]:
X_train, X_test, y_train, y_test = prepare_data(data_directory, window_size_s=1.0, split_ratio=80)
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

In [ ]:
my_model = create_model(input_shape=X_train.shape[1:], num_classes=2)
my_model.summary()

In [ ]:
# fit the model
batch_size = 32
epochs = 20

history = my_model.fit(X_train, y_train, epochs=epochs, validation_data=(X_test, y_test), batch_size=batch_size)
# plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.show()


In [ ]:
# Calculate exactly how many steps will happen
steps_per_epoch = math.ceil(len(X_train) / batch_size)
total_training_steps = steps_per_epoch * epochs
print(f"total_training_steps: {total_training_steps}, is > {len(X_train)*0.8} (len(X_train)*0.8): {total_training_steps > len(X_train)*0.8}")

In [ ]:
# --- Define a directory to save our models ---
output_dir = 'models'
os.makedirs(output_dir, exist_ok=True) # Create the directory if it doesn't exist

# Save the base model to compare size
base_model_file = os.path.join(output_dir, 'my_model.h5')
my_model.save(base_model_file, include_optimizer=False)
print(f"Base model saved to: {base_model_file}")
print(f"Base model size: {get_gzipped_model_size(base_model_file):.2f} KB")

# --- Optimization ---
print("\n--- Optimizing Model ---")
tflite_model_quant = optimize_and_convert_model(my_model, X_train, y_train, total_training_steps)
# --- Save and Compare ---
tflite_model_path = os.path.join(output_dir, 'magic_wand_model.tflite')
with open(tflite_model_path, 'wb') as f:
    f.write(tflite_model_quant)
    
print(f"Optimized (quantized) TFLite model size: {get_gzipped_model_size(tflite_model_path):.2f} KB")
print(f"TFLite model saved to: {tflite_model_path}")